In [0]:
import pandas as pd 
from matplotlib import pyplot as plt 
import seaborn as sns 
import numpy as np
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when
from databricks.feature_engineering import FeatureEngineeringClient

In [0]:
BASE_DIR = "/Volumes/workspace/default/data_customers/raw/"
train_path = f"{BASE_DIR}customer_churn_dataset-training-master.csv"
df_train = spark.read.option("header", "true").option("inferSchema", "true").csv(train_path)
display(df_train)

In [0]:
for col_name in df_train.columns:
    new_col = (
        col_name.strip()
        .lower()
        .replace(" ", "_")
        .replace("-", "_")
    )
    
    df_train = df_train.withColumnRenamed(col_name, new_col)
df_train = df_train.dropna()
df_train = df_train.dropDuplicates()
display(df_train)

In [0]:

def add_features(df):
    df = df.withColumn(
        "age_group",
        when(col("age") < 30, "young")
        .when((col("age") >= 30) & (col("age") < 50), "adult")
        .otherwise("senior"),
    )

    df = (
        df.withColumn("usage_per_tenure", col("usage_frequency") / (col("tenure") + 1))
        .withColumn("spend_per_usage", col("total_spend") / (col("usage_frequency") + 1))
        .withColumn("spend_per_tenure", col("total_spend") / (col("tenure") + 1))
        .withColumn("payment_delay_ratio", col("payment_delay") / 30)
        .withColumn("engagement_score", (col("usage_frequency") * col("total_spend")) / 1000)
    )
    return df
df_train = add_features(df_train)
display(df_train)

In [0]:
fe = FeatureEngineeringClient()

feature_df = (
    df_train
    .drop("churn")
    .dropDuplicates(["customerid"])
)

feature_table_name = "workspace.default.customer_churn_features"

fe.create_table(
    name=feature_table_name,
    primary_keys=["customerid"],
    df=feature_df,
    description="Customer churn features including demographics, usage patterns, and engineered features"
)

print(f"Feature table created: {feature_table_name}")